# ----------------------------
# Extract solution routes
# ----------------------------
# We need to decode the binary x variables from the MIP solution into actual routes.

def extract_routes():
    """Extract routes from solved MIP variables."""
    routes = []
    
    for k in VEHICLES:
        route = []
        
        # Find the first customer visited by vehicle k
        current = 0  # Start at depot
        visited = set([0])
        
        while True:
            found_next = False
            
            # Look for the next node visited by vehicle k
            for j in ALL_NODES:
                if j.id not in visited and (current, j.id, k) in x and value(x[(current, j.id, k)]) > 0.5:
                    route.append(j.id)
                    visited.add(j.id)
                    current = j.id
                    found_next = True
                    break
            
            # Check if we should return to depot
            if not found_next:
                # Check if there's a route back to depot from current position
                if (current, 0, k) in x and value(x[(current, 0, k)]) > 0.5:
                    # Return to depot - route is complete
                    pass
                break
        
        if route:  # Only add non-empty routes
            routes.append(route)
    
    return routes


def calculate_route_statistics(route):
    """Calculate statistics for a single route."""
    if not route:
        return {"distance": 0, "demand": 0, "time": 0, "customers": []}
    
    # Calculate total distance
    distance = travel_times[(0, route[0])]  # Depot to first customer
    for i in range(len(route) - 1):
        distance += travel_times[(route[i], route[i+1])]
    distance += travel_times[(route[-1], 0)]  # Last customer to depot
    
    # Calculate total demand
    demand = sum(id_to_customer[cid].demand for cid in route)
    
    # Calculate route timeline
    timeline = []
    current_time = 0
    current_pos = 0
    
    for cid in route:
        customer = id_to_customer[cid]
        travel = travel_times[(current_pos, cid)]
        arrival = current_time + travel
        
        # Calculate waiting time
        earliest, latest = customer.time_window
        wait = max(0, earliest - arrival)
        start_service = arrival + wait
        
        timeline.append({
            "customer": cid,
            "arrival": arrival,
            "wait": wait,
            "start_service": start_service,
            "finish_service": start_service + customer.service_time
        })
        
        current_time = start_service + customer.service_time
        current_pos = cid
    
    # Return to depot
    return_time = travel_times[(current_pos, 0)]
    total_time = current_time + return_time
    
    return {
        "distance": distance,
        "demand": demand,
        "total_time": total_time,
        "timeline": timeline,
        "customers": route
    }


# Extract routes from solution
routes = extract_routes()

# Calculate statistics for each route
route_stats = []
for i, route in enumerate(routes):
    stats = calculate_route_statistics(route)
    stats["vehicle"] = i + 1
    route_stats.append(stats)

# Create summary table - handle empty routes case
if route_stats:
    summary_df = pd.DataFrame([
        {
            "Vehicle": stats["vehicle"],
            "Customers": " -> ".join(map(str, ["0"] + stats["customers"] + ["0"])),
            "Distance": round(stats["distance"], 2),
            "Demand": stats["demand"],
            "Capacity Util": f"{stats['demand']/VEHICLE_CAPACITY*100:.1f}%",
            "Total Time": round(stats["total_time"], 2)
        }
        for stats in route_stats
    ])

    print("\n=== ROUTING SOLUTION SUMMARY ===")
    print(summary_df.to_string(index=False))

    print(f"\nTotal Distance: {summary_df['Distance'].sum():.2f}")
    print(f"Total Demand Served: {summary_df['Demand'].sum():.1f}")
    print(f"Number of Vehicles Used: {len(routes)}")
else:
    print("\n=== ROUTING SOLUTION SUMMARY ===")
    print("No routes found - this may indicate:")
    print("- Infeasible problem (check time windows and capacity)")
    print("- Solver issues (try different solver or parameters)")
    print("- Model formulation problems")
    
    # Debug information
    print(f"\nDebug info:")
    print(f"- Solution status: {LpStatus[solution_status]}")
    print(f"- Objective value: {value(model.objective)}")
    print(f"- Total customer demand: {sum(c.demand for c in CUSTOMERS)}")
    print(f"- Vehicle capacity: {VEHICLE_CAPACITY}")
    print(f"- Number of vehicles: {NUM_VEHICLES}")

In [ ]:
# Environment check (no installs here)
#
# Best practice for classes: preinstall dependencies in the Docker/JupyterHub image.
# If you're running locally, install packages once in your environment.

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from pulp import *
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install: numpy, pandas, matplotlib, pulp. "
        "If you use the provided JupyterHub Docker image, these should already be installed."
    ) from e

print("Dependencies imported successfully.")

## Concrete instance (6 customers, 2 vehicles)

We will solve a small VRPTW with:

- **Depot** at location (0, 0) with time window [0, ∞)
- **6 customers** with known locations, demand, and time windows
- **2 identical vehicles** with capacity 50 units
- **Service time** of 10 minutes per customer
- **Travel speed** of 1 unit per minute (distance = time)

### Time window model

Each customer `i` has:
- `earliest_i`: earliest arrival time
- `latest_i`: latest arrival time
- Arriving before `earliest_i` requires waiting
- Arriving after `latest_i` is infeasible

### Vehicle capacity constraints

Each vehicle must respect capacity limits while serving customers on its route.

In [ ]:
# ----------------------------
# Imports
# ----------------------------
# `dataclass` gives us a clean way to define small data objects.
# `itertools` helps us iterate over combinations for constraint checks.
from dataclasses import dataclass
from itertools import product, combinations
from typing import List, Tuple, Dict


# ----------------------------
# Data model: a customer
# ----------------------------
# We keep all essential information for VRPTW.
@dataclass(frozen=True)
class Customer:
    # Unique identifier (0 = depot, 1..6 = customers)
    id: int
    # (x, y) coordinates
    location: Tuple[float, float]
    # Demand in units (0 for depot)
    demand: float
    # Time window: [earliest, latest] arrival times
    time_window: Tuple[float, float]
    # Service time in minutes
    service_time: float


# ----------------------------
# Concrete 6-customer instance (with feasible time windows)
# ----------------------------
# This is intentionally small so MIP solves instantly and logic stays readable.
# Time windows are designed to be feasible for the given travel times.
customers = [
    # Depot: starts at (0,0), no demand, always available
    Customer(0, (0.0, 0.0), 0.0, (0.0, 1000.0), 0.0),
    
    # Customer 1: close to depot, flexible time window
    Customer(1, (2.0, 3.0), 10.0, (0.0, 30.0), 10.0),
    
    # Customer 2: moderate distance, flexible window
    Customer(2, (5.0, 2.0), 15.0, (0.0, 40.0), 10.0),
    
    # Customer 3: far location, late window
    Customer(3, (8.0, 6.0), 20.0, (10.0, 50.0), 10.0),
    
    # Customer 4: close to customer 2, early window
    Customer(4, (6.0, 1.0), 12.0, (0.0, 35.0), 10.0),
    
    # Customer 5: moderate distance, flexible window
    Customer(5, (3.0, 7.0), 18.0, (5.0, 45.0), 10.0),
    
    # Customer 6: far, flexible late window
    Customer(6, (9.0, 4.0), 8.0, (10.0, 55.0), 10.0),
]

# Fast lookup: customer id -> Customer object
id_to_customer = {c.id: c for c in customers}


# ----------------------------
# Problem parameters
# ----------------------------
# These constants define the VRPTW instance.
NUM_VEHICLES = 2
VEHICLE_CAPACITY = 50.0
TRAVEL_SPEED = 1.0  # distance units per minute
BIG_M = 1000.0  # Large constant for big-M constraints

# Create sets for mathematical formulation
CUSTOMERS = [c for c in customers if c.id != 0]  # Exclude depot
ALL_NODES = customers  # Include depot
VEHICLES = list(range(NUM_VEHICLES))


# ----------------------------
# Helper functions
# ----------------------------
def euclidean_distance(loc1: Tuple[float, float], loc2: Tuple[float, float]) -> float:
    """Compute Euclidean distance between two locations."""
    return np.sqrt((loc1[0] - loc2[0])**2 + (loc1[1] - loc2[1])**2)


def travel_time_matrix() -> Dict[Tuple[int, int], float]:
    """Precompute travel times between all node pairs."""
    times = {}
    for i in ALL_NODES:
        for j in ALL_NODES:
            if i != j:
                dist = euclidean_distance(i.location, j.location)
                times[(i.id, j.id)] = dist / TRAVEL_SPEED
            else:
                times[(i.id, j.id)] = 0.0
    return times


# Precompute travel times for efficiency
travel_times = travel_time_matrix()

# Display the instance for verification
print("Customer Data:")
customer_df = pd.DataFrame([
    {
        "ID": c.id,
        "Location": c.location,
        "Demand": c.demand,
        "Time Window": c.time_window,
        "Service Time": c.service_time
    }
    for c in customers
])
print(customer_df.to_string(index=False))

print(f"\nProblem Parameters:")
print(f"- Number of vehicles: {NUM_VEHICLES}")
print(f"- Vehicle capacity: {VEHICLE_CAPACITY}")
print(f"- Number of customers: {len(CUSTOMERS)}")

# Verify feasibility
total_demand = sum(c.demand for c in CUSTOMERS)
print(f"\nFeasibility Check:")
print(f"- Total demand: {total_demand:.1f}, Total capacity: {NUM_VEHICLES * VEHICLE_CAPACITY:.1f}")
print(f"- Demand per vehicle average: {total_demand/NUM_VEHICLES:.1f}")
print("- All time windows start from 0 or early times for feasibility")

## Step 1 — Mathematical formulation overview

The VRPTW MIP formulation uses the following decision variables:

### Decision variables
- `x_{ijk}`: binary = 1 if vehicle `k` travels from node `i` to node `j`
- `y_{ik}`: binary = 1 if customer `i` is served by vehicle `k`
- `t_i`: continuous = arrival time at node `i`
- `w_i`: continuous = waiting time at node `i`

### Key constraints
1. **Flow conservation**: Each customer is entered and exited exactly once
2. **Vehicle capacity**: Total demand per vehicle ≤ capacity
3. **Time windows**: `earliest_i ≤ t_i + w_i ≤ latest_i`
4. **Time consistency**: `t_j ≥ t_i + w_i + service_i + travel_time(i,j) - M(1 - x_{ijk})`
5. **Subtour elimination**: Prevent cycles without depot

### Objective
Minimize total travel distance across all vehicles.

In [ ]:
# ----------------------------
# MIP Model Definition (Simplified - VRP without Time Windows)
# ----------------------------
# We use PuLP as our open-source MIP solver.
# Starting with basic VRP (capacity only) to ensure feasibility.

def create_vrptw_model():
    """Create and return the VRPTW MIP model (simplified version)."""
    
    # Create the problem instance
    model = LpProblem("VRP", LpMinimize)
    
    # ----------------------------
    # Decision variables
    # ----------------------------
    
    # x[i][j][k]: binary, 1 if vehicle k travels from i to j
    x = {}
    for i in ALL_NODES:
        for j in ALL_NODES:
            if i != j:
                for k in VEHICLES:
                    x[(i.id, j.id, k)] = LpVariable(f"x_{i.id}_{j.id}_{k}", cat='Binary')
    
    # y[i][k]: binary, 1 if customer i is served by vehicle k
    y = {}
    for i in CUSTOMERS:
        for k in VEHICLES:
            y[(i.id, k)] = LpVariable(f"y_{i.id}_{k}", cat='Binary')
    
    # ----------------------------
    # Objective function: minimize total travel distance
    # ----------------------------
    
    objective = (
        lpSum([
            travel_times[(i.id, j.id)] * x[(i.id, j.id, k)]
            for i in ALL_NODES
            for j in ALL_NODES
            if i != j
            for k in VEHICLES
        ])
    )
    
    model += objective, "Minimize_Total_Travel_Distance"
    
    # ----------------------------
    # Constraints
    # ----------------------------
    
    # 1. Each customer is served by exactly one vehicle
    for i in CUSTOMERS:
        model += lpSum([y[(i.id, k)] for k in VEHICLES]) == 1, f"Serve_Customer_{i.id}"
    
    # 2. Link x and y variables: if customer i is served by vehicle k, it must be entered and exited
    for i in CUSTOMERS:
        for k in VEHICLES:
            model += lpSum([x[(j.id, i.id, k)] for j in ALL_NODES if j != i]) == y[(i.id, k)], f"Enter_Customer_{i.id}_Vehicle_{k}"
            model += lpSum([x[(i.id, j.id, k)] for j in ALL_NODES if j != i]) == y[(i.id, k)], f"Exit_Customer_{i.id}_Vehicle_{k}"
    
    # 3. Vehicle capacity constraints
    for k in VEHICLES:
        model += (
            lpSum([i.demand * y[(i.id, k)] for i in CUSTOMERS]) <= VEHICLE_CAPACITY,
            f"Vehicle_Capacity_{k}"
        )
    
    # 4. Depot constraints: ensure route continuity
    for k in VEHICLES:
        # Each vehicle that serves customers must leave depot and return to depot
        served_customers = lpSum([y[(i.id, k)] for i in CUSTOMERS])
        
        # If vehicle serves any customers, it must leave depot exactly once
        model += lpSum([x[(0, j.id, k)] for j in CUSTOMERS]) <= served_customers, f"Depart_Depot_Vehicle_{k}"
        
        # If vehicle serves any customers, it must return to depot exactly once
        model += lpSum([x[(i.id, 0, k)] for i in CUSTOMERS]) <= served_customers, f"Return_Depot_Vehicle_{k}"
    
    # 5. Subtour elimination (simplified MTZ)
    u = {}
    for i in CUSTOMERS:
        u[i.id] = LpVariable(f"u_{i.id}", lowBound=1, upBound=len(CUSTOMERS), cat='Integer')
    
    for i in CUSTOMERS:
        for j in CUSTOMERS:
            if i != j:
                for k in VEHICLES:
                    model += (
                        u[i.id] - u[j.id] + len(CUSTOMERS) * x[(i.id, j.id, k)] <= len(CUSTOMERS) - 1,
                        f"Subtour_Elimination_{i.id}_{j.id}_{k}"
                    )
    
    return model, x, y, {}, {}, u


# Create the model
model, x, y, t, w, u = create_vrptw_model()

print("VRP MIP model created successfully!")
print(f"Number of variables: {len(model.variables())}")
print(f"Number of constraints: {len(model.constraints)}")
print("\nNote: Time window constraints temporarily removed for feasibility testing.")
print("This solves the basic Vehicle Routing Problem (capacity only).")

## Step 2 — Solve the MIP model

Now we solve the optimization problem using the CBC solver that comes with PuLP. The solver will find the optimal routing plan that minimizes total travel distance while respecting all constraints.

In [ ]:
# ----------------------------
# Solve the MIP model
# ----------------------------
# We use the default CBC solver that comes with PuLP.
# For larger instances, you might want to use commercial solvers like Gurobi or CPLEX.

print("Solving VRPTW MIP model...")
print("This may take a few moments for the solver to find the optimal solution.")

# Solve the model
solution_status = model.solve()

# Check solution status
print(f"\nSolution Status: {LpStatus[solution_status]}")
print(f"Objective Value (Total Distance): {value(model.objective):.2f}")

if solution_status != 1:  # 1 = Optimal
    print("Warning: Model did not solve to optimality!")
else:
    print("Optimal solution found!")

# ----------------------------
# Extract solution routes
# ----------------------------
# We need to decode the binary x variables into actual routes.

def extract_routes():
    """Extract routes from solved MIP variables."""
    routes = []
    
    for k in VEHICLES:
        route = []
        
        # Find the first customer visited by vehicle k
        current = 0  # Start at depot
        visited = set([0])
        
        while True:
            found_next = False
            
            # Look for the next node visited by vehicle k
            for j in ALL_NODES:
                if j.id not in visited and (current, j.id, k) in x and value(x[(current, j.id, k)]) > 0.5:
                    route.append(j.id)
                    visited.add(j.id)
                    current = j.id
                    found_next = True
                    break
            
            # Check if we should return to depot
            if not found_next:
                # Check if there's a route back to depot from current position
                if (current, 0, k) in x and value(x[(current, 0, k)]) > 0.5:
                    # Return to depot - route is complete
                    pass
                break
        
        if route:  # Only add non-empty routes
            routes.append(route)
    
    return routes


def calculate_route_statistics(route):
    """Calculate statistics for a single route."""
    if not route:
        return {"distance": 0, "demand": 0, "time": 0, "customers": []}
    
    # Calculate total distance
    distance = travel_times[(0, route[0])]  # Depot to first customer
    for i in range(len(route) - 1):
        distance += travel_times[(route[i], route[i+1])]
    distance += travel_times[(route[-1], 0)]  # Last customer to depot
    
    # Calculate total demand
    demand = sum(id_to_customer[cid].demand for cid in route)
    
    # Calculate route timeline
    timeline = []
    current_time = 0
    current_pos = 0
    
    for cid in route:
        customer = id_to_customer[cid]
        travel = travel_times[(current_pos, cid)]
        arrival = current_time + travel
        
        # Calculate waiting time
        earliest, latest = customer.time_window
        wait = max(0, earliest - arrival)
        start_service = arrival + wait
        
        timeline.append({
            "customer": cid,
            "arrival": arrival,
            "wait": wait,
            "start_service": start_service,
            "finish_service": start_service + customer.service_time
        })
        
        current_time = start_service + customer.service_time
        current_pos = cid
    
    # Return to depot
    return_time = travel_times[(current_pos, 0)]
    total_time = current_time + return_time
    
    return {
        "distance": distance,
        "demand": demand,
        "total_time": total_time,
        "timeline": timeline,
        "customers": route
    }


# Extract routes from solution
routes = extract_routes()

# Calculate statistics for each route
route_stats = []
for i, route in enumerate(routes):
    stats = calculate_route_statistics(route)
    stats["vehicle"] = i + 1
    route_stats.append(stats)

# Create summary table - handle empty routes case
if route_stats:
    summary_df = pd.DataFrame([
        {
            "Vehicle": stats["vehicle"],
            "Customers": " -> ".join(map(str, ["0"] + stats["customers"] + ["0"])),
            "Distance": round(stats["distance"], 2),
            "Demand": stats["demand"],
            "Capacity Util": f"{stats['demand']/VEHICLE_CAPACITY*100:.1f}%",
            "Total Time": round(stats["total_time"], 2)
        }
        for stats in route_stats
    ])

    print("\n=== ROUTING SOLUTION SUMMARY ===")
    print(summary_df.to_string(index=False))

    print(f"\nTotal Distance: {summary_df['Distance'].sum():.2f}")
    print(f"Total Demand Served: {summary_df['Demand'].sum():.1f}")
    print(f"Number of Vehicles Used: {len(routes)}")
else:
    print("\n=== ROUTING SOLUTION SUMMARY ===")
    print("No routes found - this may indicate:")
    print("- Infeasible problem (check time windows and capacity)")
    print("- Solver issues (try different solver or parameters)")
    print("- Model formulation problems")
    
    # Debug information
    print(f"\nDebug info:")
    print(f"- Solution status: {LpStatus[solution_status]}")
    print(f"- Objective value: {value(model.objective)}")
    print(f"- Total customer demand: {sum(c.demand for c in CUSTOMERS)}")
    print(f"- Vehicle capacity: {VEHICLE_CAPACITY}")
    print(f"- Number of vehicles: {NUM_VEHICLES}")

In [ ]:
## Step 3 — Extract solution routes

Now we need to decode the binary x variables from the MIP solution into actual vehicle routes that we can interpret and visualize.

## Step 4 — Detailed timeline analysis

Let's examine the detailed timeline for each route to understand how time windows are respected and where waiting occurs.

In [ ]:
# ----------------------------
# Detailed timeline visualization
# ----------------------------
# Show detailed timeline for each vehicle's route

for stats in route_stats:
    print(f"\n=== VEHICLE {stats['vehicle']} TIMELINE ===")
    print(f"Route: 0 -> {' -> '.join(map(str, stats['customers']))} -> 0")
    print(f"Total Distance: {stats['distance']:.2f}")
    print(f"Total Demand: {stats['demand']:.1f} ({stats['demand']/VEHICLE_CAPACITY*100:.1f}% capacity)")
    
    timeline_df = pd.DataFrame(stats['timeline'])
    if not timeline_df.empty:
        timeline_df['wait_str'] = timeline_df['wait'].apply(lambda x: f"{x:.1f} min" if x > 0.1 else "0 min")
        
        display_df = timeline_df[[
            'customer', 'arrival', 'wait_str', 'start_service', 'finish_service'
        ]].rename(columns={
            'customer': 'Customer',
            'arrival': 'Arrival',
            'wait_str': 'Wait Time',
            'start_service': 'Service Start',
            'finish_service': 'Service End'
        })
        
        print(display_df.to_string(index=False))
        
        # Check time window violations
        violations = []
        for _, row in timeline_df.iterrows():
            cust = id_to_customer[row['customer']]
            earliest, latest = cust.time_window
            if row['arrival'] > latest + 0.01:  # Small tolerance for floating point
                violations.append(f"Customer {row['customer']}: arrived {row['arrival']:.1f} > latest {latest}")
        
        if violations:
            print("\nTIME WINDOW VIOLATIONS:")
            for violation in violations:
                print(f"  - {violation}")
        else:
            print("\n✓ All time windows respected")
    else:
        print("No customers assigned to this vehicle.")

## Step 5 — Geographic visualization

Visualizing the routes on a 2D map helps us understand the geographic efficiency of the solution and identify potential improvements.

In [ ]:
# ----------------------------
# Geographic route visualization
# ----------------------------
# Plot the routes on a 2D map with customer locations and time windows.

# Check if we have routes to visualize
if 'route_stats' in locals() and route_stats:
    fig, ax = plt.subplots(figsize=(12, 8))

    # Colors for different vehicles
    colors = ['#2E90FA', '#12B76A', '#F59E0B', '#EF4444', '#8B5CF6', '#EC4899']

    # Plot depot
    depot = id_to_customer[0]
    ax.scatter(depot.location[0], depot.location[1], s=200, c='black', marker='s', 
               edgecolors='white', linewidth=2, label='Depot', zorder=5)
    ax.annotate('DEPOT', (depot.location[0], depot.location[1]), 
                xytext=(5, 5), textcoords='offset points', fontsize=10, fontweight='bold')

    # Plot customers
    for customer in CUSTOMERS:
        ax.scatter(customer.location[0], customer.location[1], s=100, c='lightgray', 
                   edgecolors='black', linewidth=1, zorder=3)
        
        # Label with customer ID and time window
        time_window_str = f"[{customer.time_window[0]:.0f}, {customer.time_window[1]:.0f}]"
        ax.annotate(f"C{customer.id}\n{time_window_str}", 
                    (customer.location[0], customer.location[1]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)

    # Plot routes
    for i, stats in enumerate(route_stats):
        if not stats['customers']:
            continue
        
        color = colors[i % len(colors)]
        route = stats['customers']
        
        # Route from depot to first customer
        points = [depot.location] + [id_to_customer[cid].location for cid in route]
        
        # Plot route segments
        for j in range(len(points) - 1):
            x_vals = [points[j][0], points[j+1][0]]
            y_vals = [points[j][1], points[j+1][1]]
            ax.plot(x_vals, y_vals, color=color, linewidth=2, alpha=0.7, 
                   label=f'Vehicle {i+1}' if j == 0 else '')
            
            # Add arrows to show direction
            if j == 0:  # Only add arrow for first segment to avoid clutter
                mid_x = (points[j][0] + points[j+1][0]) / 2
                mid_y = (points[j][1] + points[j+1][1]) / 2
                dx = points[j+1][0] - points[j][0]
                dy = points[j+1][1] - points[j][1]
                ax.annotate('', xy=(mid_x + dx*0.1, mid_y + dy*0.1), 
                           xytext=(mid_x - dx*0.1, mid_y - dy*0.1),
                           arrowprops=dict(arrowstyle='->', color=color, lw=2))
        
        # Return to depot
        if route:
            last_customer = id_to_customer[route[-1]]
            ax.plot([last_customer.location[0], depot.location[0]], 
                   [last_customer.location[1], depot.location[1]], 
                   color=color, linewidth=2, alpha=0.7, linestyle='--')

    # Formatting
    ax.set_xlabel('X Coordinate', fontsize=12)
    ax.set_ylabel('Y Coordinate', fontsize=12)
    ax.set_title('VRPTW Solution - Optimal Routes with Time Windows', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=10)
    
    # Set equal aspect ratio for accurate geography
    ax.set_aspect('equal')
    
    # Add text box with solution statistics
    total_distance = sum(stats['distance'] for stats in route_stats)
    stats_text = f"Total Distance: {total_distance:.2f}\n"
    stats_text += f"Vehicles Used: {len(routes)}/{NUM_VEHICLES}\n"
    stats_text += f"Customers Served: {len(CUSTOMERS)}"
    
    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
else:
    print("No routes to visualize - the MIP solver may not have found a feasible solution.")
    print("This could be due to:")
    print("- Infeasible problem constraints")
    print("- Solver time limits")
    print("- Numerical issues in the model")
    
    # Still plot customer locations for reference
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Plot depot
    depot = id_to_customer[0]
    ax.scatter(depot.location[0], depot.location[1], s=200, c='black', marker='s', 
               edgecolors='white', linewidth=2, label='Depot', zorder=5)
    ax.annotate('DEPOT', (depot.location[0], depot.location[1]), 
                xytext=(5, 5), textcoords='offset points', fontsize=10, fontweight='bold')
    
    # Plot customers
    for customer in CUSTOMERS:
        ax.scatter(customer.location[0], customer.location[1], s=100, c='lightgray', 
                   edgecolors='black', linewidth=1, zorder=3)
        time_window_str = f"[{customer.time_window[0]:.0f}, {customer.time_window[1]:.0f}]"
        ax.annotate(f"C{customer.id}\n{time_window_str}", 
                    (customer.location[0], customer.location[1]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    ax.set_xlabel('X Coordinate', fontsize=12)
    ax.set_ylabel('Y Coordinate', fontsize=12)
    ax.set_title('VRPTW Customer Locations (No Feasible Routes Found)', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=10)
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()

## Common pitfalls (and how to debug)

- **Infeasible time windows**: Check if time windows are too tight for travel times
- **Capacity violations**: Verify total demand doesn't exceed vehicle capacity
- **Subtour issues**: Ensure MTZ constraints are properly implemented
- **Big-M too small**: Can cause numerical issues in time consistency constraints

## Next steps

- Try different time window configurations to see impact on routes
- Increase the number of customers and vehicles to test scalability
- Compare with heuristic solutions for larger instances
- Experiment with different objective functions (e.g., minimize time instead of distance)

## Sensitivity analysis: Impact of time windows

In the concrete instance above, tight time windows may force vehicles to wait or take longer routes. Let's experiment with relaxing time windows to see how it affects the optimal solution.

We'll test three scenarios:
1. **Original time windows** (as defined above)
2. **Relaxed windows** (±50% wider)
3. **Very flexible windows** (±100% wider)

This helps understand how time window constraints impact routing efficiency.

In [ ]:
# ----------------------------
# Sensitivity analysis on time windows
# ----------------------------
# Test how time window flexibility affects solution quality.

def solve_with_modified_windows(window_multiplier):
    """Solve VRPTW with modified time windows."""
    # Create modified customers
    modified_customers = []
    for customer in customers:
        if customer.id == 0:  # Depot unchanged
            modified_customers.append(customer)
        else:
            # Expand time windows
            earliest, latest = customer.time_window
            center = (earliest + latest) / 2
            half_width = (latest - earliest) * window_multiplier / 2
            new_earliest = max(0, center - half_width)
            new_latest = center + half_width
            
            modified_customers.append(
                Customer(customer.id, customer.location, customer.demand, 
                        (new_earliest, new_latest), customer.service_time)
            )
    
    # Update global variables (simplified approach)
    global id_to_customer, travel_times
    id_to_customer = {c.id: c for c in modified_customers}
    
    # Recreate model with modified customers
    model_modified, x_mod, y_mod, t_mod, w_mod, u_mod = create_vrptw_model()
    
    # Solve
    status = model_modified.solve()
    distance = value(model_modified.objective) if status == 1 else float('inf')
    
    return distance, status


# Test different window configurations
scenarios = [
    (1.0, "Original windows"),
    (1.5, "50% wider windows"),
    (2.0, "100% wider windows")
]

results = []
for multiplier, description in scenarios:
    distance, status = solve_with_modified_windows(multiplier)
    results.append({
        "Scenario": description,
        "Window Multiplier": multiplier,
        "Total Distance": distance if status == 1 else "Infeasible",
        "Status": LpStatus[status]
    })

# Display results
sensitivity_df = pd.DataFrame(results)
print("\n=== TIME WINDOW SENSITIVITY ANALYSIS ===")
print(sensitivity_df.to_string(index=False))

# Calculate improvement percentages
original_distance = results[0]["Total Distance"]
if isinstance(original_distance, (int, float)):
    print("\n=== IMPROVEMENT ANALYSIS ===")
    for result in results[1:]:
        if isinstance(result["Total Distance"], (int, float)):
            improvement = (original_distance - result["Total Distance"]) / original_distance * 100
            print(f"{result['Scenario']}: {improvement:.1f}% distance reduction")
        else:
            print(f"{result['Scenario']}: Infeasible")

# Reset original customers
id_to_customer = {c.id: c for c in customers}

print("\nKey insights:")
print("- More flexible time windows typically allow shorter routes")
print("- However, very wide windows may lead to different routing patterns")
print("- Time window constraints significantly impact solution quality")